In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [ ]:
# Google Drive 上的檔案路徑
file_path = '/content/drive/MyDrive/Colab Notebooks/training.csv'

# 設定檔案路徑和批次大小
#file_path = 'your_12GB_file.csv'
chunk_size = 1000
n_chunks_to_process = 50  # 處理前幾個 chunks 作為範例，您可以處理更多或全部

In [ ]:
# 定義特徵和目標變數
features = [
    '外資券商_分點買賣力', '外資券商_前1天分點買賣力',
    '官股券商_持股比率(%)',
    '個股主力買賣超統計_近5日主力買賣超(%)',
    '日外資_外資買賣超', '日投信_投信買賣超',
    '技術指標_RSI(5)', '技術指標_MACD',
    '月營收_單月營收年成長(%)', '季IFRS財報_毛利率(%)',
    # 根據您的資料選擇更多特徵
]
target = '飆股'

In [ ]:
# 初始化模型和預處理器
model = SGDClassifier(loss='log_loss', random_state=42, learning_rate='adaptive', eta0=0.01)
scaler = StandardScaler()
imputer = SimpleImputer(strategy='mean')

# 追蹤第一個批次以進行初始擬合
first_chunk = True

In [ ]:
#import pandas as pd
#file_path = 'your_12GB_file.csv'
df_head = pd.read_csv(file_path, nrows=1000)  # 讀取前 100 行
print(df_head[target].unique())

[0 1]


In [ ]:

# 初始化增量學習模型：SGDClassifier
# loss='log_loss'：用於二元分類的邏輯迴歸損失函數
# random_state=42：確保結果的可重複性
# learning_rate='adaptive'：學習率會根據訓練進度自動調整
# eta0=0.01：初始學習率
model = SGDClassifier(loss='log_loss', random_state=42, learning_rate='adaptive', eta0=0.01)

# 初始化特徵縮放器：StandardScaler
# 用於將特徵縮放到具有零均值和單位變異數的範圍內，有助於模型訓練
scaler = StandardScaler()

# 初始化缺失值填充器：SimpleImputer
# 用於處理資料中的缺失值。這裡使用平均值填充，您可以嘗試其他策略，例如中位數或最頻繁值。
imputer = SimpleImputer(strategy='mean')

# 標記是否是第一個批次。第一個批次需要用於擬合 (fit) 預處理器。
first_chunk = True

# 逐批讀取和處理資料
for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size)):
    print(f"處理批次 {i+1}，大小：{len(chunk)}")

    # 檢查目標變數是否存在於當前批次中
    if target not in chunk.columns:
        print(f"目標變數 '{target}' 不存在於批次 {i+1} 中，跳過此批次。")
        continue

    # 檢查所有指定的特徵欄位是否存在於當前批次中
    missing_features_in_chunk = [f for f in features if f not in chunk.columns]
    if missing_features_in_chunk:
        print(f"以下特徵不存在於批次 {i+1} 中：{missing_features_in_chunk}，跳過此批次。")
        continue

    # 分割當前批次的特徵 (X_chunk) 和目標變數 (y_chunk)
    X_chunk = chunk[features]
    y_chunk = chunk[target]

    #print("if 處理缺失值")
    # 處理缺失值
    if first_chunk:
        # 對第一個批次擬合 (fit) imputer，並轉換資料
        X_chunk = imputer.fit_transform(X_chunk)
    else:
        # 對後續批次僅進行轉換 (transform)
        X_chunk = imputer.transform(X_chunk)

    #print("if 特徵縮放")
    # 特徵縮放
    if first_chunk:
        # 對第一個批次擬合 (fit) scaler，並轉換資料
        X_chunk = scaler.fit_transform(X_chunk)
    else:
        # 對後續批次僅進行轉換 (transform)
        X_chunk = scaler.transform(X_chunk)

    # 增量訓練模型
    # partial_fit 方法用於逐步訓練模型。
    # classes 參數需要指定所有可能的類別標籤。對於二元分類，通常是 [0, 1] 或 ['否', '是'] 等。
    # 我們在這裡假設目標變數的類別在整個資料集中是一致的，並在第一個批次中獲取所有唯一值。
    if first_chunk:
        model.partial_fit(X_chunk, y_chunk, classes=y_chunk.unique())
        first_chunk = False
    else:
        model.partial_fit(X_chunk, y_chunk)

    # 限制處理的批次數量 (用於演示)
    #print("if n_chunks_to_process")
    if n_chunks_to_process is not None and i >= n_chunks_to_process - 1:
        break

print("\n增量模型訓練完成！")

# 評估模型 (可選：使用部分保留的測試資料進行評估)
# 為了節省記憶體，這裡僅使用最後一個批次作為一個非常小的測試集進行演示。
# 在實際應用中，您應該使用一個獨立的、具有代表性的測試集進行更可靠的評估。
# 或者，您可以在每個批次訓練後，使用該批次的部分資料進行臨時評估。
try:
    # 重新讀取最後一個處理的批次作為測試集 (僅用於演示)
    test_chunk = pd.read_csv(file_path, skiprows=range(1, i * chunk_size + 1), nrows=chunk_size)
    if target in test_chunk.columns and all(f in test_chunk.columns for f in features):
        X_test = test_chunk[features]
        y_test = test_chunk[target]

        # 使用相同的 imputer 和 scaler 對測試集進行轉換
        X_test = imputer.transform(X_test)
        X_test = scaler.transform(X_test)

        # 在測試集上進行預測
        y_pred = model.predict(X_test)

        # 評估模型性能
        accuracy = accuracy_score(y_test, y_pred)
        print(f"\n在最後一個批次（作為小型測試集）上的準確率：{accuracy:.4f}")
        print("\n分類報告：")
        print(classification_report(y_test, y_pred))
    else:
        print("\n無法載入足夠的資料進行評估。")

except Exception as e:
    print(f"\n評估過程中發生錯誤：{e}")

# 對新的資料進行預測可以使用類似的分批處理方式
# for new_chunk in pd.read_csv('new_large_file.csv', chunksize=chunk_size):
#     if all(f in new_chunk.columns for f in features):
#         X_new_chunk = imputer.transform(new_chunk[features])
#         X_new_chunk = scaler.transform(X_new_chunk)
#         predictions = model.predict(X_new_chunk)
#         # 處理預測結果
#     else:
#         print("新資料批次缺少必要的特徵。")

處理批次 1，大小：1000
處理批次 2，大小：1000
處理批次 3，大小：1000
處理批次 4，大小：1000
處理批次 5，大小：1000
處理批次 6，大小：1000
處理批次 7，大小：1000
處理批次 8，大小：1000
處理批次 9，大小：1000
處理批次 10，大小：1000
處理批次 11，大小：1000
處理批次 12，大小：1000
處理批次 13，大小：1000
處理批次 14，大小：1000
處理批次 15，大小：1000
處理批次 16，大小：1000
處理批次 17，大小：1000
處理批次 18，大小：1000
處理批次 19，大小：1000
處理批次 20，大小：1000
處理批次 21，大小：1000
處理批次 22，大小：1000
處理批次 23，大小：1000
處理批次 24，大小：1000
處理批次 25，大小：1000
處理批次 26，大小：1000
處理批次 27，大小：1000
處理批次 28，大小：1000
處理批次 29，大小：1000
處理批次 30，大小：1000
處理批次 31，大小：1000
處理批次 32，大小：1000
處理批次 33，大小：1000
處理批次 34，大小：1000
處理批次 35，大小：1000
處理批次 36，大小：1000
處理批次 37，大小：1000
處理批次 38，大小：1000
處理批次 39，大小：1000
處理批次 40，大小：1000
處理批次 41，大小：1000
處理批次 42，大小：1000
處理批次 43，大小：1000
處理批次 44，大小：1000
處理批次 45，大小：1000
處理批次 46，大小：1000
處理批次 47，大小：1000
處理批次 48，大小：1000
處理批次 49，大小：1000
處理批次 50，大小：1000

增量模型訓練完成！

在最後一個批次（作為小型測試集）上的準確率：0.9970

分類報告：
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       997
           1       0.00      0.00      0.00         

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# 設定要預測的檔案路徑
predict_file_path = '/content/drive/MyDrive/Colab Notebooks/public_x.csv'
output_prediction_file = '/content/drive/MyDrive/Colab Notebooks/predictions_public.csv'

In [ ]:
# 載入您的訓練好的模型、縮放器和填充器
loaded_model = model
loaded_scaler = scaler
loaded_imputer = imputer

predict_chunk_size = 1000  # 設定預測時的批次大小

# 設定用於預測的特徵欄位名稱 (必須與訓練時使用的特徵一致)

# 準備儲存預測結果的列表
all_predictions = []
all_ids = []

# 逐批讀取 public_x.csv 檔案並進行預測
try:
    for chunk in pd.read_csv(predict_file_path, chunksize=predict_chunk_size):
        print(f"處理預測批次，大小：{len(chunk)}")

        # 確保當前批次包含所有需要的特徵
        missing_features = [f for f in features if f not in chunk.columns]
        if missing_features:
            print(f"警告：預測批次缺少以下特徵：{missing_features}，跳過此批次。")
            continue

        # 提取 ID 和特徵資料
        ids = chunk['ID'].tolist()
        X_predict_chunk = chunk[features]

        # 處理缺失值
        X_predict_chunk_imputed = loaded_imputer.transform(X_predict_chunk)

        # 進行特徵縮放
        X_predict_chunk_scaled = loaded_scaler.transform(X_predict_chunk_imputed)

        # 使用訓練好的模型進行預測
        predictions_chunk = loaded_model.predict(X_predict_chunk_scaled)

        all_predictions.extend(predictions_chunk)
        all_ids.extend(ids)

    # 建立包含 ID 和預測結果的 DataFrame
    predictions_df = pd.DataFrame({'ID': all_ids, '飆股': all_predictions})

    # 將結果儲存到新的 CSV 檔案
    try:
        predictions_df.to_csv(output_prediction_file, index=False)
        print(f"預測結果已成功儲存至：{output_prediction_file}")
    except Exception as e:
        print(f"儲存預測結果時發生錯誤：{e}")

except FileNotFoundError:
    print(f"找不到預測檔案：{predict_file_path}，請確認檔案路徑是否正確。")
except Exception as e:
    print(f"讀取預測檔案時發生錯誤：{e}")

處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：1000
處理預測批次，大小：108
預測結果已成功儲存至：/content/drive/MyDrive/Colab Notebooks/predictions_public.csv


In [ ]:

#chunk_size = 1000
#n_chunks_to_process = 50
# 設定儲存到 Google Drive 的路徑
output_file_path_gdrive = '/content/drive/MyDrive/Colab Notebooks/predictions.csv'  # 儲存到 Google Drive 的根目錄

# 輸出包含 "ID" 和預測結果的 CSV 檔案到 Google Drive
#predictions_df = pd.DataFrame({'ID': X_test.index, '結果': y_pred})
try:
    predictions_df.to_csv(output_file_path_gdrive, index=False)
    print(f"\n預測結果已成功儲存至 Google Drive：{output_file_path_gdrive}")
except Exception as e:
    print(f"儲存預測結果到 Google Drive 時發生錯誤：{e}")

In [ ]:
#2025.04.22（二）15:30
#已提升類別1的f1分數，但犧牲一些類別0的f1。

import pandas as pd
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from collections import Counter
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Google Drive 上的檔案路徑
file_path = '/content/drive/MyDrive/Colab Notebooks/training.csv'

# 設定檔案路徑和批次大小
#file_path = 'your_12GB_file.csv'
chunk_size = 1000
n_chunks_to_process = 100  # 處理前幾個 chunks 作為範例，您可以處理更多或全部

# 定義特徵和目標變數
features = [
    '外資券商_分點買賣力', '外資券商_前1天分點買賣力',
    '官股券商_持股比率(%)',
    '個股主力買賣超統計_近5日主力買賣超(%)',
    '日外資_外資買賣超', '日投信_投信買賣超',
    '技術指標_RSI(5)', '技術指標_MACD',
    '月營收_單月營收年成長(%)', '季IFRS財報_毛利率(%)',
    # 根據您的資料選擇更多特徵
]
target = '飆股'

positive_class = 1
all_classes = np.array([0, 1])  # 所有可能的類別

# 初始化類別權重 (在迴圈之前，使用第一個批次估計)
estimated_class_weights = None
first_weight_estimation_chunk = True

# 初始化模型和預處理器
model = PassiveAggressiveClassifier(random_state=42, class_weight=None)
scaler = StandardScaler()
imputer = SimpleImputer(strategy='mean')

# 標記是否是第一個批次
first_chunk = True

# 逐批讀取和處理資料
for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size)):
    print(f"處理訓練批次 {i+1}，大小：{len(chunk)}")

    if target not in chunk.columns or not all(f in chunk.columns for f in features):
        print("目標變數或部分特徵不存在於當前批次中，跳過此批次。")
        continue

    X_chunk = chunk[features]
    y_chunk = chunk[target]

    # 處理缺失值
    X_chunk = imputer.fit_transform(X_chunk) if first_chunk else imputer.transform(X_chunk)

    # 標準化特徵
    X_chunk = scaler.fit_transform(X_chunk) if first_chunk else scaler.transform(X_chunk)

    # 估計類別權重 (僅使用第一個批次)
    if first_chunk and first_weight_estimation_chunk:
        class_weights = compute_class_weight('balanced', classes=all_classes, y=y_chunk)
        estimated_class_weights = dict(zip(all_classes, class_weights))
        model.class_weight = estimated_class_weights  # 將權重賦給模型
        print(f"使用第一個批次估計的類別權重並設定到模型：{estimated_class_weights}")
        first_weight_estimation_chunk = False

    # 簡單的過採樣 (僅在當前批次內，使用估計的權重調整過採樣比例)
    if not first_chunk and estimated_class_weights is not None:
        positive_indices = y_chunk[y_chunk == positive_class].index
        if len(positive_indices) > 0:
            positive_weight = estimated_class_weights.get(positive_class, 1.0)
            negative_weight = estimated_class_weights.get(0, 1.0)
            sampling_rate = negative_weight / positive_weight if positive_weight > 0 else 1.0
            num_to_oversample = int(len(y_chunk[y_chunk == positive_class]) * sampling_rate)
            oversampled_indices = np.random.choice(positive_indices, size=num_to_oversample, replace=True)
            oversampled_X = X_chunk[oversampled_indices]
            oversampled_y = y_chunk.iloc[oversampled_indices]
            X_chunk = np.vstack([X_chunk, oversampled_X])
            y_chunk = pd.concat([y_chunk, oversampled_y])
            print(f"  過採樣後，批次大小：{len(y_chunk)}, 類別分佈：{Counter(y_chunk)}")
        else:
            print("  當前批次沒有正類別樣本，跳過過採樣。")
    elif first_chunk:
        print(f"  第一個批次，類別分佈：{Counter(y_chunk)}")

    # 增量訓練模型
    model.partial_fit(X_chunk, y_chunk, classes=all_classes)

    first_chunk = False

    if n_chunks_to_process is not None and i >= n_chunks_to_process - 1:
        break

print("\n增量模型訓練完成！")

# 後續您可以添加評估程式碼

from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
'''
# 設定檔案路徑和批次大小 (與訓練時相同)
file_path = 'your_12GB_file.csv'  # 請替換為您的實際檔案路徑
chunk_size = 100000

# 定義特徵和目標變數 (與訓練時相同)
features = [
    '外資券商_分點買賣力', '主力券商_分點買賣力',
    '官股券商_買賣超', '官股券商_持股比率(%)',
    '個股主力買賣超統計_近5日主力買賣超(%)',
    '日外資_外資買賣超', '日投信_投信買賣超',
    '技術指標_RSI(5)', '技術指標_MACD',
    '月營收_單月營收年成長(%)', '季IFRS財報_毛利率(%)',
    # 根據您的資料選擇更多特徵
]
target = '預測是否為"飆股"'
'''
all_classes = np.array([0, 1])  # 所有可能的類別

n_test_chunks = 20  # 設定要使用的測試批次數量

try:
    # 計算總批次數
    total_chunks = 200
    if total_chunks < n_test_chunks:
        print(f"警告：檔案總批次數 ({total_chunks}) 小於要測試的批次數 ({n_test_chunks})。將使用所有剩餘批次進行測試。")
        start_chunk_index = 0
    else:
        start_chunk_index = total_chunks - n_test_chunks

    all_y_true = []
    all_y_pred = []

    print(f"從第 {start_chunk_index + 1} 個批次開始讀取測試資料...")

    # 逐批讀取測試資料
    for i, test_chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size, skiprows=range(1, start_chunk_index * chunk_size + 1))):
        print(f"處理測試批次 {i + 1}，大小：{len(test_chunk)}")

        if target in test_chunk.columns and all(f in test_chunk.columns for f in features):
            X_test = test_chunk[features]
            y_test = test_chunk[target]

            # 使用相同的 imputer 和 scaler 對測試集進行轉換
            X_test = imputer.transform(X_test)
            X_test = scaler.transform(X_test)

            # 在測試集上進行預測
            y_pred = model.predict(X_test)

            all_y_true.extend(y_test)
            all_y_pred.extend(y_pred)

            if i >= n_test_chunks - 1 or (total_chunks < n_test_chunks and i >= total_chunks - 1):
                print("已處理完所有測試批次。")
                break
        else:
            print(f"警告：測試批次 {i + 1} 缺少目標變數或部分特徵，跳過。")

    if all_y_true:
        # 評估模型在所有測試批次上的性能
        accuracy = accuracy_score(all_y_true, all_y_pred)
        print(f"\n在最後 {len(all_y_true) // len(test_chunk) if len(test_chunk) > 0 else 0} 個批次上的總體準確率：{accuracy:.4f}")
        print("\n總體分類報告：")
        print(classification_report(all_y_true, all_y_pred))
    else:
        print("\n沒有可用的測試資料用於評估。")

except FileNotFoundError:
    print(f"找不到檔案：{file_path}，請確認檔案路徑是否正確。")
except Exception as e:
    print(f"\n評估過程中發生錯誤：{e}")

處理訓練批次 1，大小：1000
使用第一個批次估計的類別權重並設定到模型：{np.int64(0): np.float64(0.5035246727089627), np.int64(1): np.float64(71.42857142857143)}
  第一個批次，類別分佈：Counter({0: 993, 1: 7})
處理訓練批次 2，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 996, 1: 4})
處理訓練批次 3，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 989, 1: 11})
處理訓練批次 4，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 990, 1: 10})
處理訓練批次 5，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 993, 1: 7})
處理訓練批次 6，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 995, 1: 5})
處理訓練批次 7，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 996, 1: 4})
處理訓練批次 8，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 991, 1: 9})
處理訓練批次 9，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 994, 1: 6})
處理訓練批次 10，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 994, 1: 6})
處理訓練批次 11，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 996, 1: 4})
處理訓練批次 12，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 997, 1: 3})
處理訓練批次 13，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 992, 1: 8})
處理訓練批次 14，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 995, 1: 

In [ ]:
#2025.04.22（二）18:30
#已提升類別1的f1分數，但犧牲一些類別0的f1。

import pandas as pd
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from collections import Counter
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Google Drive 上的檔案路徑
file_path = '/content/drive/MyDrive/Colab Notebooks/training.csv'

# 設定檔案路徑和批次大小
#file_path = 'your_12GB_file.csv'
chunk_size = 1000
n_chunks_to_process = 200  # 處理前幾個 chunks 作為範例，您可以處理更多或全部

# 定義特徵和目標變數
features = [
    '外資券商_分點買賣力', '外資券商_前1天分點買賣力',
    '外資券商_前2天分點買賣力', '外資券商_前3天分點買賣力',
    '外資券商_前4天分點買賣力', '外資券商_前5天分點買賣力',
    '官股券商_持股比率(%)',
    '個股主力買賣超統計_近5日主力買賣超(%)',
    '日外資_外資買賣超', '日投信_投信買賣超',
    '技術指標_RSI(5)', '技術指標_MACD',
    '月營收_單月營收年成長(%)', '季IFRS財報_毛利率(%)',
    # 根據您的資料選擇更多特徵
]
target = '飆股'

positive_class = 1
all_classes = np.array([0, 1])  # 所有可能的類別

# 初始化類別權重 (在迴圈之前，使用第一個批次估計)
estimated_class_weights = None
first_weight_estimation_chunk = True

# 初始化模型和預處理器
model = PassiveAggressiveClassifier(random_state=42, class_weight=None)
scaler = StandardScaler()
imputer = SimpleImputer(strategy='mean')

# 標記是否是第一個批次
first_chunk = True

# 逐批讀取和處理資料
for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size)):
    print(f"處理訓練批次 {i+1}，大小：{len(chunk)}")

    if target not in chunk.columns or not all(f in chunk.columns for f in features):
        print("目標變數或部分特徵不存在於當前批次中，跳過此批次。")
        continue

    X_chunk = chunk[features]
    y_chunk = chunk[target]

    # 處理缺失值
    X_chunk = imputer.fit_transform(X_chunk) if first_chunk else imputer.transform(X_chunk)

    # 標準化特徵
    X_chunk = scaler.fit_transform(X_chunk) if first_chunk else scaler.transform(X_chunk)

    # 估計類別權重 (僅使用第一個批次)
    if first_chunk and first_weight_estimation_chunk:
        class_weights = compute_class_weight('balanced', classes=all_classes, y=y_chunk)
        estimated_class_weights = dict(zip(all_classes, class_weights))
        model.class_weight = estimated_class_weights  # 將權重賦給模型
        print(f"使用第一個批次估計的類別權重並設定到模型：{estimated_class_weights}")
        first_weight_estimation_chunk = False

    # 簡單的過採樣 (僅在當前批次內，使用估計的權重調整過採樣比例)
    if not first_chunk and estimated_class_weights is not None:
        positive_indices = y_chunk[y_chunk == positive_class].index
        if len(positive_indices) > 0:
            positive_weight = estimated_class_weights.get(positive_class, 1.0)
            negative_weight = estimated_class_weights.get(0, 1.0)
            sampling_rate = negative_weight / positive_weight if positive_weight > 0 else 1.0
            num_to_oversample = int(len(y_chunk[y_chunk == positive_class]) * sampling_rate)
            oversampled_indices = np.random.choice(positive_indices, size=num_to_oversample, replace=True)
            oversampled_X = X_chunk[oversampled_indices]
            oversampled_y = y_chunk.iloc[oversampled_indices]
            X_chunk = np.vstack([X_chunk, oversampled_X])
            y_chunk = pd.concat([y_chunk, oversampled_y])
            print(f"  過採樣後，批次大小：{len(y_chunk)}, 類別分佈：{Counter(y_chunk)}")
        else:
            print("  當前批次沒有正類別樣本，跳過過採樣。")
    elif first_chunk:
        print(f"  第一個批次，類別分佈：{Counter(y_chunk)}")

    # 增量訓練模型
    model.partial_fit(X_chunk, y_chunk, classes=all_classes)

    first_chunk = False

    if n_chunks_to_process is not None and i >= n_chunks_to_process - 1:
        break

print("\n增量模型訓練完成！")

# 後續您可以添加評估程式碼

from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
'''
# 設定檔案路徑和批次大小 (與訓練時相同)
file_path = 'your_12GB_file.csv'  # 請替換為您的實際檔案路徑
chunk_size = 100000

# 定義特徵和目標變數 (與訓練時相同)
features = [
    '外資券商_分點買賣力', '主力券商_分點買賣力',
    '官股券商_買賣超', '官股券商_持股比率(%)',
    '個股主力買賣超統計_近5日主力買賣超(%)',
    '日外資_外資買賣超', '日投信_投信買賣超',
    '技術指標_RSI(5)', '技術指標_MACD',
    '月營收_單月營收年成長(%)', '季IFRS財報_毛利率(%)',
    # 根據您的資料選擇更多特徵
]
target = '預測是否為"飆股"'
'''
all_classes = np.array([0, 1])  # 所有可能的類別

n_test_chunks = 20  # 設定要使用的測試批次數量

try:
    # 計算總批次數
    total_chunks = 200
    if total_chunks < n_test_chunks:
        print(f"警告：檔案總批次數 ({total_chunks}) 小於要測試的批次數 ({n_test_chunks})。將使用所有剩餘批次進行測試。")
        start_chunk_index = 0
    else:
        start_chunk_index = total_chunks - n_test_chunks

    all_y_true = []
    all_y_pred = []

    print(f"從第 {start_chunk_index + 1} 個批次開始讀取測試資料...")

    # 逐批讀取測試資料
    for i, test_chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size, skiprows=range(1, start_chunk_index * chunk_size + 1))):
        print(f"處理測試批次 {i + 1}，大小：{len(test_chunk)}")

        if target in test_chunk.columns and all(f in test_chunk.columns for f in features):
            X_test = test_chunk[features]
            y_test = test_chunk[target]

            # 使用相同的 imputer 和 scaler 對測試集進行轉換
            X_test = imputer.transform(X_test)
            X_test = scaler.transform(X_test)

            # 在測試集上進行預測
            y_pred = model.predict(X_test)

            all_y_true.extend(y_test)
            all_y_pred.extend(y_pred)

            if i >= n_test_chunks - 1 or (total_chunks < n_test_chunks and i >= total_chunks - 1):
                print("已處理完所有測試批次。")
                break
        else:
            print(f"警告：測試批次 {i + 1} 缺少目標變數或部分特徵，跳過。")

    if all_y_true:
        # 評估模型在所有測試批次上的性能
        accuracy = accuracy_score(all_y_true, all_y_pred)
        print(f"\n在最後 {len(all_y_true) // len(test_chunk) if len(test_chunk) > 0 else 0} 個批次上的總體準確率：{accuracy:.4f}")
        print("\n總體分類報告：")
        print(classification_report(all_y_true, all_y_pred))
    else:
        print("\n沒有可用的測試資料用於評估。")

except FileNotFoundError:
    print(f"找不到檔案：{file_path}，請確認檔案路徑是否正確。")
except Exception as e:
    print(f"\n評估過程中發生錯誤：{e}")

處理訓練批次 1，大小：1000
使用第一個批次估計的類別權重並設定到模型：{np.int64(0): np.float64(0.5035246727089627), np.int64(1): np.float64(71.42857142857143)}
  第一個批次，類別分佈：Counter({0: 993, 1: 7})
處理訓練批次 2，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 996, 1: 4})
處理訓練批次 3，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 989, 1: 11})
處理訓練批次 4，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 990, 1: 10})
處理訓練批次 5，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 993, 1: 7})
處理訓練批次 6，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 995, 1: 5})
處理訓練批次 7，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 996, 1: 4})
處理訓練批次 8，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 991, 1: 9})
處理訓練批次 9，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 994, 1: 6})
處理訓練批次 10，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 994, 1: 6})
處理訓練批次 11，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 996, 1: 4})
處理訓練批次 12，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 997, 1: 3})
處理訓練批次 13，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 992, 1: 8})
處理訓練批次 14，大小：1000
  過採樣後，批次大小：1000, 類別分佈：Counter({0: 995, 1: 

In [6]:
# 設定要預測的檔案路徑
predict_file_path = '/content/drive/MyDrive/Colab Notebooks/private_x.csv'
output_prediction_file = '/content/drive/MyDrive/Colab Notebooks/predictions_private2.csv'
# 載入您的訓練好的模型、縮放器和填充器
loaded_model = model
loaded_scaler = scaler
loaded_imputer = imputer

features = [
    '外資券商_分點買賣力', '外資券商_前1天分點買賣力',
    '外資券商_前2天分點買賣力', '外資券商_前3天分點買賣力',
    '外資券商_前4天分點買賣力', '外資券商_前5天分點買賣力',
    '官股券商_持股比率(%)',
    '個股主力買賣超統計_近5日主力買賣超(%)',
    '日外資_外資買賣超', '日投信_投信買賣超',
    '技術指標_RSI(5)', '技術指標_MACD',
    '月營收_單月營收年成長(%)', '季IFRS財報_毛利率(%)',
    # 根據您的資料選擇更多特徵
]
target = '飆股'

predict_chunk_size = 1000  # 設定預測時的批次大小

# 設定用於預測的特徵欄位名稱 (必須與訓練時使用的特徵一致)

# 準備儲存預測結果的列表
all_predictions = []
all_ids = []

# 逐批讀取 public_x.csv 檔案並進行預測
try:
    for chunk in pd.read_csv(predict_file_path, chunksize=predict_chunk_size):
        print(f"處理預測批次，大小：{len(chunk)}")

        # 確保當前批次包含所有需要的特徵
        missing_features = [f for f in features if f not in chunk.columns]
        if missing_features:
            print(f"警告：預測批次缺少以下特徵：{missing_features}，跳過此批次。")
            continue

        # 提取 ID 和特徵資料
        ids = chunk['ID'].tolist()
        X_predict_chunk = chunk[features]

        # 處理缺失值
        X_predict_chunk_imputed = loaded_imputer.transform(X_predict_chunk)

        # 進行特徵縮放
        X_predict_chunk_scaled = loaded_scaler.transform(X_predict_chunk_imputed)

        # 使用訓練好的模型進行預測
        predictions_chunk = loaded_model.predict(X_predict_chunk_scaled)

        all_predictions.extend(predictions_chunk)
        all_ids.extend(ids)

    # 建立包含 ID 和預測結果的 DataFrame
    predictions_df = pd.DataFrame({'ID': all_ids, '飆股': all_predictions})

    # 將結果儲存到新的 CSV 檔案
    try:
        predictions_df.to_csv(output_prediction_file, index=False)
        print(f"預測結果已成功儲存至：{output_prediction_file}")
    except Exception as e:
        print(f"儲存預測結果時發生錯誤：{e}")

except FileNotFoundError:
    print(f"找不到預測檔案：{predict_file_path}，請確認檔案路徑是否正確。")
except Exception as e:
    print(f"讀取預測檔案時發生錯誤：{e}")

NameError: name 'model' is not defined

In [7]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 檔案路徑 (請根據您的實際路徑修改)
train_file_path = 'training.csv'
predict_file_path = 'public_x.csv'
output_prediction_file = 'xgb_predictions_public_quarter.csv'

# 定義特徵和目標變數
features = [
    '外資券商_分點買賣力', '外資券商_前1天分點買賣力',
    '官股券商_持股比率(%)',
    '個股主力買賣超統計_近5日主力買賣超(%)',
    '日外資_外資買賣超', '日投信_投信買賣超',
    '技術指標_RSI(5)', '技術指標_MACD',
    '月營收_單月營收年成長(%)', '季IFRS財報_毛利率(%)',
    # 根據您的資料選擇更多特徵 (與您之前的程式碼保持一致)
]
target = '飆股'

# 載入訓練資料
try:
    train_df = pd.read_csv(train_file_path)
    print(f"成功載入完整訓練資料，大小：{train_df.shape}")
except FileNotFoundError:
    print(f"找不到訓練檔案：{train_file_path}，請確認檔案路徑是否正確。")
    exit()

# 取四分之一的訓練資料
train_df_quarter = train_df.sample(frac=0.25, random_state=42)
print(f"使用四分之一的訓練資料，大小：{train_df_quarter.shape}")

# 檢查目標變數和特徵是否存在
if target not in train_df_quarter.columns or not all(f in train_df_quarter.columns for f in features):
    print("目標變數或部分特徵不存在於四分之一的訓練資料中，請檢查欄位名稱。")
    exit()

X_train = train_df_quarter[features]
y_train = train_df_quarter[target]

# 處理缺失值
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)

# 特徵縮放
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)

# 初始化 XGBoost 分類器
model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,  # 避免 FutureWarning
    random_state=42
)

# 訓練模型
print("開始使用四分之一的資料訓練 XGBoost 模型...")
model.fit(X_train_scaled, y_train)
print("XGBoost 模型訓練完成！")

# 載入預測資料
try:
    predict_df = pd.read_csv(predict_file_path)
    print(f"成功載入預測資料，大小：{predict_df.shape}")
except FileNotFoundError:
    print(f"找不到預測檔案：{predict_file_path}，請確認檔案路徑是否正確。")
    exit()

# 確保預測資料包含所有需要的特徵
missing_features_predict = [f for f in features if f not in predict_df.columns]
if missing_features_predict:
    print(f"警告：預測資料缺少以下特徵：{missing_features_predict}，預測結果可能不準確。")

X_predict = predict_df[features]

# 處理預測資料的缺失值和縮放
X_predict_imputed = imputer.transform(X_predict)
X_predict_scaled = scaler.transform(X_predict_imputed)

# 進行預測
print("開始預測 public_x.csv...")
predictions_proba = model.predict_proba(X_predict_scaled)[:, 1]  # 取得預測為 1 的機率
predictions = (predictions_proba > 0.5).astype(int)  # 將機率轉換為 0 或 1

# 建立包含 ID 和預測結果的 DataFrame
predictions_df = pd.DataFrame({'ID': predict_df['ID'], '飆股': predictions})

# 將預測結果儲存到 CSV 檔案 (與程式碼同一個資料夾)
try:
    predictions_df.to_csv(output_prediction_file, index=False)
    print(f"預測結果已成功儲存至：{output_prediction_file}")
except Exception as e:
    print(f"儲存預測結果時發生錯誤：{e}")

找不到訓練檔案：training.csv，請確認檔案路徑是否正確。


NameError: name 'train_df' is not defined